# 04 · Paper 1 follow-up: does the VLM fail to read, or fail to answer?

Qwen2-VL-2B on the **same 200 test questions** and the **same 150-DPI page images** as the original run, scored with the official TAT-DQA scorer. Conditions:
- **A** original run (96 output tokens, prompt chosen by the gold answer type), loaded from file, no GPU
- **B** same prompts, **512** output tokens: does a longer budget fix arithmetic?
- **C** one **generic** prompt for every question (no gold information), 512 tokens
- **D** **structured** output: `Answer:` (value, or an arithmetic expression executed by Python) + `Unit:`

Runtime → Change runtime type → **T4 GPU** (L4/A100 is faster). Rough T4 time: 1.5–2.5 h in total; every condition saves to Drive and resumes after a disconnect (re-run setup + model cell, then the same cell). At the end, send `paper1_rerun_results.json` from `My Drive/finsight/runs/`.

In [ ]:
# --- Setup: on Colab, gets the latest code + data from GitHub; also works on a local GPU ---
import os, sys, subprocess, shutil, random
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less GPU memory fragmentation
IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    ROOT = "/content/finnn"
    shutil.rmtree(ROOT, ignore_errors=True)               # always fetch the latest code
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/praseeda-cse/finnn.git", ROOT], check=True)
    try:                                                  # save results on Drive so they survive disconnects
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = "/content/drive/MyDrive/finsight/runs"
    except Exception as e:
        print("Drive not mounted (", e, ") - results are saved in /content/runs and are LOST if Colab disconnects")
        OUT = "/content/runs"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate"], check=True)
else:
    ROOT = os.path.abspath("..")
    OUT = os.path.join(ROOT, "runs")
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, ROOT)

import json, torch
from finsight.data import load_split
from finsight.evaluate import score, print_report
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - select a GPU runtime!")

docs_train, q_train = load_split(f"{ROOT}/data/tatdqa_train.json.gz")
docs_dev, q_dev = load_split(f"{ROOT}/data/tatdqa_dev.json.gz")
docs_test, q_test = load_split(f"{ROOT}/data/tatdqa_test.json.gz")
def sample(questions, n, seed=0):
    return random.Random(seed).sample(questions, n) if n < len(questions) else questions
def report_row(r):
    return {m: v for m, v in r.items() if m != "rows"}
print(len(q_train), len(q_dev), len(q_test), "questions")

In [ ]:
# Page images (150 DPI, as in the original run) + original outputs.
# Upload tatdqa_test_pages_150dpi.zip (83 MB, from the laptop: Desktop/oneee/final_year/) to My Drive/finsight/
import zipfile
PAGES = "/content/tatdqa_pages" if IN_COLAB else os.path.join(ROOT, "runs", "tatdqa_pages")
ZIP = "/content/drive/MyDrive/finsight/tatdqa_test_pages_150dpi.zip"
if not os.path.exists(f"{PAGES}/page_of_doc.json"):
    assert os.path.exists(ZIP), "Upload tatdqa_test_pages_150dpi.zip to My Drive/finsight/ first"
    zipfile.ZipFile(ZIP).extractall(PAGES)
page_of = json.load(open(f"{PAGES}/page_of_doc.json"))
old_raw = json.load(open(f"{PAGES}/old_run_raw.json", encoding="utf-8"))

N = 200
eval_qs = sample(q_test, N)                       # fixed sample (seed 0)
image_of = {q["uid"]: f"{PAGES}/pages/{q['doc_uid']}_{page_of[q['doc_uid']]}.png" for q in eval_qs}
gold_scale = {q["uid"]: q["scale"] for q in eval_qs}
print(len(eval_qs), "questions,", sum(os.path.exists(p) for p in image_of.values()), "page images found")

In [ ]:
from finsight import vlm
model, processor = vlm.load()
# quick check on 2 questions before the long runs
print(vlm.run(model, processor, eval_qs[:2], image_of, "structured", "/tmp/check.jsonl", 64, batch_size=2))

In [ ]:
RAW = {"A": {q["uid"]: old_raw[q["uid"]] for q in eval_qs}}
RAW["B"] = vlm.run(model, processor, eval_qs, image_of, "orig", f"{OUT}/p1_B_orig512.jsonl", 512)
RAW["C"] = vlm.run(model, processor, eval_qs, image_of, "generic", f"{OUT}/p1_C_generic512.jsonl", 512)
RAW["D"] = vlm.run(model, processor, eval_qs, image_of, "structured", f"{OUT}/p1_D_structured.jsonl", 128)

In [ ]:
MODE = {"A": "orig", "B": "orig", "C": "generic", "D": "structured"}
NAME = {"A": "original (96 tokens, gold-type prompt)", "B": "512 tokens, gold-type prompt",
        "C": "512 tokens, generic prompt", "D": "structured: expression executed + unit"}
SUMMARY = {}
for k in "ABCD":
    p = vlm.to_preds(RAW[k], MODE[k])
    r = score(eval_qs, p)
    r_gold = score(eval_qs, vlm.to_preds(RAW[k], MODE[k], gold_scale))
    print_report(r, f"{k}: {NAME[k]}")
    SUMMARY[k] = {"name": NAME[k], "official": report_row(r), "gold_scale": report_row(r_gold),
                  "arithmetic_behaviour": vlm.behaviour(eval_qs, RAW[k], p),
                  "examples": [{"q": q["question"], "type": q["answer_type"], "gold": q["answer"], "scale": q["scale"],
                                "raw": RAW[k][q["uid"]][:600]} for q in eval_qs[:15]]}
    print("   arithmetic behaviour:", SUMMARY[k]["arithmetic_behaviour"], "\n")
SUMMARY["gpu"] = torch.cuda.get_device_name(0)
json.dump(SUMMARY, open(f"{OUT}/paper1_rerun_results.json", "w"), indent=1, ensure_ascii=False)
print("saved", f"{OUT}/paper1_rerun_results.json")

print("\nEM (official) / EM with gold unit:")
for k in "ABCD":
    s = SUMMARY[k]
    print(f"  {k}  {s['official']['EM']:5.1f} / {s['gold_scale']['EM']:5.1f}   {s['name']}")